# Google Colab — CNN6 de 6 clases (CirCor + SPRSound) y destilación

Dos usos del mismo script (`run_cnn6_gpu.py`), con el mismo paquete de datos:

- `MODEL = "cnn6"`: fine-tuning de la CNN6 preentrenada de PANNs como modelo único
  de 6 clases, con la receta de `cs_wide` (`avances/26`). Ganó `ours64`.
- `MODEL = "unified"`: **destilación** (`avances/26` §7.1). El alumno es la
  `UnifiedCNN` de `cs_wide` (ancho `base`, `wide` o `large`) y el maestro es la CNN6
  `ours64` de cada fold, que corre en la GPU sobre la misma onda aumentada.
  Con `ALPHA = 1` no hay maestro: es el control sin destilar.

**Qué hace cada cosa:**
- **Código**: se clona del repositorio de GitHub (la última versión subida).
- **Datos**: un único archivo `colab_cnn6_circor_spr.zip` (~1.15 GB) en tu Drive:
  el audio a 4 kHz de CirCor y SPRSound, los índices de ventanas, los folds y los
  pesos de CNN6. Se copia una sola vez al disco local de la VM y el audio entero
  se carga en la GPU; el espectrograma se calcula en la GPU.
- **Maestro** (solo destilación): una carpeta de Drive con los `fold{i}_best.pt` y
  `result_fold{i}.json` de `cnn6_ours64`. Con los `result_fold`, cada fold controla
  que el maestro reproduzca su validación antes de entrenar.
- **Resultados**: se guardan en tu Drive, época por época.

**Antes de correr:**
1. Subir el código nuevo a GitHub (`git push`).
2. Subir `desarrollo/data/colab_cnn6_circor_spr.zip` (lo genera
   `unified_model/pack_cnn6_colab.py`) a la carpeta de Drive de la configuración.
3. Para destilar: subir a `TEACHER_DIR` los 5 `fold{i}_best.pt` de
   `desarrollo/src/unified_model/checkpoints_cnn6_ours64/` y los 5
   `result_fold{i}.json` de `desarrollo/reports/unified_model/runs/cnn6_ours64/`.
4. Activar la GPU: `Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU`.

**Si la sesión se corta**: volver a correr todas las celdas. Retoma desde la
última época guardada en Drive y no repite los folds terminados.

In [ ]:
print("=== Montando Google Drive ===")
from google.colab import drive
drive.mount('/content/drive')

## Configuración

**Plan de la destilación** (`avances/26` §7.1), filtro en los folds 0 y 3 (`FOLDS = "0 3"`):

1. Con `WIDTH = "wide"`: el control `ALPHA = 1` y las tres combinaciones
   (`ALPHA`, `TEMPERATURE`) = (0.5, 2), (0.5, 4) y (0.1, 4). Una corrida por vez.
2. Localmente, `filter_variants.py` compara con el control y con `cs_wide`, solo
   en validación. Pasa la que supera al control en la media y no baja en ningún fold.
3. Con la mejor combinación, `WIDTH = "base"` y `"large"`, cada uno con su control.
4. **5 folds** de la ganadora: `FOLDS = "0 1 2 3 4"` (los folds 0 y 3 ya
   terminados se saltean).

Cada corrida guarda en su carpeta dentro de `OUT_DIR`: `kd_<ancho>_a<α·100>_t<T>`
(p. ej. `kd_wide_a50_t2`), `gpu_<ancho>` para el control y `cnn6_<frontend>` para la CNN6.

In [ ]:
from pathlib import Path

# Editá estas rutas si usás otra carpeta de Drive.
DRIVE_DIR   = Path("/content/drive/MyDrive/CEIA/colab_cnn6")
PACKAGE_ZIP = DRIVE_DIR / "colab_cnn6_circor_spr.zip"
OUT_DIR     = DRIVE_DIR / "resultados"
TEACHER_DIR = DRIVE_DIR / "maestro_cnn6_ours64"   # fold{i}_best.pt + result_fold{i}.json

MODEL       = "unified"   # "unified" (destilación) o "cnn6"
WIDTH       = "wide"      # alumno: "base" (24k), "wide" (95k) o "large" (374k)
ALPHA       = 0.5         # peso de la CE con etiquetas; 1 = control sin destilar
TEMPERATURE = 2
FRONTEND    = "ours64"    # solo MODEL = "cnn6"
FOLDS       = "0 3"       # filtro: "0 3"; 5 folds: "0 1 2 3 4"

REPO_URL = "https://github.com/SantiagoBettig/deteccion_patologias_cardiopulmonares.git"
REPO_DIR = Path("/content/repo")
PKG_DIR  = Path("/content/pkg")

assert PACKAGE_ZIP.exists(), f"No se encontró el paquete de datos: {PACKAGE_ZIP}"
OUT_DIR.mkdir(parents=True, exist_ok=True)
if MODEL == "cnn6":
    RUN_ARGS = f"--model cnn6 --frontend {FRONTEND}"
else:
    RUN_ARGS = f"--model unified --width {WIDTH} --alpha {ALPHA} --temperature {TEMPERATURE}"
    if ALPHA < 1:
        n_ckpt = len(list(TEACHER_DIR.glob("fold*_best.pt")))
        n_res = len(list(TEACHER_DIR.glob("result_fold*.json")))
        assert n_ckpt == 5, f"Faltan checkpoints del maestro en {TEACHER_DIR} ({n_ckpt} de 5)"
        print(f"✓ Maestro: {TEACHER_DIR} ({n_ckpt} checkpoints, {n_res} result_fold para el control)")
        RUN_ARGS += f" --teacher-dir {TEACHER_DIR}"
print(f"✓ Paquete: {PACKAGE_ZIP} ({PACKAGE_ZIP.stat().st_size / 1e6:.0f} MB)")
print(f"✓ Resultados en: {OUT_DIR}")
print(f"✓ {RUN_ARGS} | folds: {FOLDS}")

In [ ]:
print("=== Verificando GPU ===")
import torch
print("CUDA disponible:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Activá la GPU: Entorno de ejecución → Cambiar tipo de entorno de ejecución"
print("Dispositivo:", torch.cuda.get_device_name(0))

In [ ]:
print("=== Clonando / actualizando el repositorio ===")
import subprocess
if (REPO_DIR / ".git").exists():
    print(subprocess.run(["git", "-C", str(REPO_DIR), "pull"], capture_output=True, text=True).stdout)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
SRC_DIR = REPO_DIR / "desarrollo" / "src"
assert (SRC_DIR / "unified_model" / "run_cnn6_gpu.py").exists(),     "El repositorio no tiene run_cnn6_gpu.py: ¿hiciste git push del código nuevo?"
print(subprocess.run(["git", "-C", str(REPO_DIR), "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
print("=== Dependencias ===")
# torch, numpy, pandas, scikit-learn, matplotlib, scipy y librosa ya vienen en Colab.
# No se instala requirements.txt para no arriesgar reemplazar el torch con GPU de Colab.
import librosa, sklearn, pandas, numpy, scipy
print("✓ librosa", librosa.__version__, "| torch", torch.__version__)

In [ ]:
print("=== Copiando el paquete de datos al disco local de la VM ===")
import json, shutil, time, zipfile
if not (PKG_DIR / "audio_4k_int16.npy").exists():
    t0 = time.time()
    local_zip = Path("/content/package.zip")
    shutil.copy2(PACKAGE_ZIP, local_zip)
    zipfile.ZipFile(local_zip).extractall(PKG_DIR)
    local_zip.unlink()
    print(f"✓ Copiado y descomprimido en {time.time() - t0:.0f} s")
print(sorted(p.name for p in PKG_DIR.iterdir()))
print(json.dumps(json.loads((PKG_DIR / "manifest.json").read_text()), indent=1))

## Entrenamiento

Una línea por época: loss de train (en la destilación, también la CE y la KD por
separado) / val, AUC de val (murmullo, crackle, wheeze), accuracy de dominio,
criterios A y B, segundos. `*` = mejoró el criterio principal (B), `+` = mejoró el
otro (A). Con maestro, cada fold empieza con su control en val (tiene que coincidir
con el esperado). Al terminar cada fold, el resultado de test por ventana. Si la
sesión se corta, volver a correr desde arriba: retoma solo.

In [ ]:
!cd {SRC_DIR} && python -u unified_model/run_cnn6_gpu.py     --package-dir {PKG_DIR} --out-dir {OUT_DIR} {RUN_ARGS} --folds {FOLDS}

## Después de entrenar

Los resultados quedan en `OUT_DIR/<run>/` en tu Drive. Bajar esa carpeta (entera,
con los `.pt`, que hacen falta para la validación externa) y, desde `desarrollo/src`:

```
python unified_model/import_colab_run.py --src <carpeta bajada>/gpu_wide
python unified_model/import_colab_run.py --src <carpeta bajada>/kd_wide_a50_t2
python unified_model/filter_variants.py --runs cs_wide gpu_wide kd_wide_a50_t2 kd_wide_a50_t4 kd_wide_a10_t4
```

Con los 5 folds de la ganadora:

```
python unified_model/import_colab_run.py --src <carpeta bajada>/<run>
python unified_model/evaluate_circor_spr.py --run <run> --base-run cs_wide --gate-run kfold_cs
python unified_model/plot_circor_spr.py --run <run> --base-run cs_wide
```